
# Case Study — Multi-Asset Ledger-Aware Backtest (Inspired by nouveau 2)

This notebook builds a refined **3-symbol case study** on **NVDA, AMZN, AAPL** using the current backtest stack:

- Multi-asset event-driven engine
- Realistic execution frictions (fees/slippage/spread)
- Portfolio constraints (weights, notional/exposure controls)
- **Accounting ledger outputs** (per-symbol cost basis, realized/unrealized PnL, journal, portfolio exposure)

Compared with `nouveau 2.ipynb`, this version is structured as a full research case-study with richer diagnostics and accounting-aware visualization.

This refresh also demonstrates the new **per-symbol stop-loss + cooldown** controls in `EngineConfig`.


In [ ]:

# Optional: use autoreload if working interactively
%load_ext autoreload
%autoreload 2


In [ ]:

from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analytics.tearsheet import make_tearsheet_with_details
from data.loaders.yfinance_loader import load_yfinance
from engine.core.backtest_engine import BacktestEngine, EngineConfig
from engine.core.event import MarketEvent


## 1) Universe + configuration (with stop-loss + cooldown)


In [ ]:
SYMBOLS = ["NVDA", "AMZN", "AAPL"]
PERIOD = "2y"
INTERVAL = "1d"

cfg = EngineConfig(
    initial_cash=150_000,
    fee_bps=1.0,
    slippage_bps=2.0,
    spread_bps=1.0,
    latency_bars=0,
    max_abs_weight=0.60,
    max_abs_exposure=1.0,
    max_notional=120_000,
    allow_short=True,
    borrow_rate_bps=30.0,
    financing_bars_per_year=252,
    stop_loss_mode="pct",
    stop_loss_value=0.08,
    stop_cooldown_bars=5,
)

print("Risk controls:")
print({
    "max_abs_weight": cfg.max_abs_weight,
    "max_abs_exposure": cfg.max_abs_exposure,
    "max_notional": cfg.max_notional,
    "stop_loss_mode": cfg.stop_loss_mode,
    "stop_loss_value": cfg.stop_loss_value,
    "stop_cooldown_bars": cfg.stop_cooldown_bars,
})


## 2) Load and align the 3-symbol universe

In [ ]:

def load_universe(symbols: list[str], period: str, interval: str) -> list[MarketEvent]:
    bars_by_symbol: dict[str, list[MarketEvent]] = {}
    for symbol in symbols:
        bars_by_symbol[symbol] = load_yfinance(symbol=symbol, period=period, interval=interval, auto_adjust=True)

    common_timestamps = None
    for symbol, bars in bars_by_symbol.items():
        ts = {b.timestamp for b in bars}
        common_timestamps = ts if common_timestamps is None else (common_timestamps & ts)

    if not common_timestamps:
        return []

    common_timestamps = set(common_timestamps)
    merged: list[MarketEvent] = []
    for symbol in symbols:
        filtered = [b for b in bars_by_symbol[symbol] if b.timestamp in common_timestamps]
        merged.extend(filtered)

    merged.sort(key=lambda x: (x.timestamp, x.symbol))
    return merged

bars = load_universe(SYMBOLS, PERIOD, INTERVAL)
print(f"Loaded bars: {len(bars)}")
print(f"First bar: {bars[0] if bars else 'None'}")


## 3) Strategy: adaptive trend + pullback blend (creative variant)

In [ ]:

@dataclass
class AdaptiveTrendPullback:
    lookback_mom: int = 30
    lookback_vol: int = 20
    pullback_window: int = 10
    top_k: int = 2

    def __post_init__(self) -> None:
        self.prices: dict[str, deque[float]] = defaultdict(lambda: deque(maxlen=max(self.lookback_mom + 1, self.lookback_vol + 2, self.pullback_window + 2)))

    def _ret(self, series: list[float], n: int) -> float:
        if len(series) <= n or series[-n-1] == 0:
            return 0.0
        return (series[-1] / series[-n-1]) - 1.0

    def _vol(self, series: list[float], n: int) -> float:
        if len(series) < n + 1:
            return 0.0
        rets = []
        for i in range(-n, 0):
            prev = series[i-1]
            cur = series[i]
            rets.append((cur / prev) - 1.0 if prev else 0.0)
        return float(np.std(rets))

    def on_bars(self, bars_by_symbol: dict[str, MarketEvent]) -> dict[str, float]:
        scores: list[tuple[str, float, float]] = []

        for symbol, bar in bars_by_symbol.items():
            self.prices[symbol].append(bar.close)
            s = list(self.prices[symbol])
            if len(s) < max(self.lookback_mom + 1, self.lookback_vol + 1, self.pullback_window + 1):
                continue

            mom = self._ret(s, self.lookback_mom)
            vol = self._vol(s, self.lookback_vol)
            pullback = self._ret(s, self.pullback_window)

            # composite score:
            # - trend positive (mom)
            # - penalize high vol
            # - mild contrarian boost when recent pullback is negative
            score = mom - 0.5 * vol + 0.3 * (-pullback)
            scores.append((symbol, score, vol))

        if len(scores) < self.top_k:
            return {symbol: 0.0 for symbol in bars_by_symbol}

        scores.sort(key=lambda x: x[1], reverse=True)
        longs = scores[: self.top_k]
        shorts = scores[-self.top_k :]

        # inverse-vol risk scaling
        def inv_vol(v: float) -> float:
            return 1.0 / max(v, 1e-6)

        long_scale = sum(inv_vol(v) for _, _, v in longs)
        short_scale = sum(inv_vol(v) for _, _, v in shorts)

        targets = {symbol: 0.0 for symbol in bars_by_symbol}
        for symbol, _, vol in longs:
            targets[symbol] += 0.5 * (inv_vol(vol) / long_scale)
        for symbol, _, vol in shorts:
            targets[symbol] -= 0.5 * (inv_vol(vol) / short_scale)

        return targets

strategy = AdaptiveTrendPullback()


## 4) Run backtest and compute headline metrics

In [ ]:

engine = BacktestEngine(cfg)
result = engine.run_detailed(bars, strategy)

# representative price series aligned to timestamps (use first symbol per timestamp)
bars_df = pd.DataFrame([{"timestamp": b.timestamp, "symbol": b.symbol, "close": b.close} for b in bars])
price_ref = bars_df.sort_values(["timestamp", "symbol"]).groupby("timestamp", as_index=False).first()
prices = price_ref["close"].tolist()[: len(result.positions)]

metrics = make_tearsheet_with_details(
    equity_curve=result.equity_curve,
    annualization=252,
    fills=result.fills,
    positions=result.positions,
    prices=prices,
    rolling_window=20,
)

pd.Series(metrics).sort_index()


## 5) Build ledger dataframes

In [ ]:

fills_df = pd.DataFrame([
    {
        "timestamp": f.timestamp,
        "symbol": f.symbol,
        "side": f.side,
        "quantity": f.quantity,
        "fill_price": f.fill_price,
        "fee": f.fee,
    }
    for f in result.fills
])

sym_df = pd.DataFrame(result.positions_by_symbol)
port_df = pd.DataFrame(result.portfolio_history)
journal_df = pd.DataFrame(result.journal)
trade_attr_df = pd.DataFrame(result.trade_attribution)

for df in [sym_df, port_df, journal_df, trade_attr_df, fills_df]:
    if not df.empty and "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"])

print("fills:", len(fills_df))
print("symbol snapshots:", len(sym_df))
print("portfolio snapshots:", len(port_df))
print("journal entries:", len(journal_df))
print("trade attribution rows:", len(trade_attr_df))


## 6) Plot A — Equity + drawdown (kept and refined)

In [ ]:

eq = pd.Series(result.equity_curve, name="equity")
roll_max = eq.cummax()
drawdown = (eq / roll_max) - 1.0

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

axes[0].plot(eq.values, label="Equity", color="tab:blue")
axes[0].plot(roll_max.values, label="Rolling peak", color="tab:gray", alpha=0.6)
axes[0].set_title("Equity Curve and Rolling Peak")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].fill_between(range(len(drawdown)), drawdown.values, 0, color="tab:red", alpha=0.3)
axes[1].plot(drawdown.values, color="tab:red", linewidth=1)
axes[1].set_title("Drawdown")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## 7) Plot B — Symbol-level market value exposure from ledger

In [ ]:

if not sym_df.empty:
    expo = sym_df.pivot_table(index="timestamp", columns="symbol", values="market_value", aggfunc="last").fillna(0.0)

    ax = expo.plot(figsize=(14, 5), linewidth=1.5)
    ax.set_title("Ledger Market Value by Symbol")
    ax.set_ylabel("Market Value")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("No symbol snapshot data available")


## 8) Plot C — Realized vs unrealized PnL by symbol (ledger refinement)

In [ ]:

if not sym_df.empty:
    last_snap = sym_df.sort_values("timestamp").groupby("symbol", as_index=False).tail(1)
    pnl_plot = last_snap[["symbol", "realized_pnl", "unrealized_pnl"]].set_index("symbol")

    ax = pnl_plot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Ending Realized vs Unrealized PnL by Symbol")
    ax.set_ylabel("PnL")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

    display(last_snap[["symbol", "quantity", "avg_cost", "last_price", "realized_pnl", "unrealized_pnl", "total_pnl"]].sort_values("total_pnl", ascending=False))
else:
    print("No symbol snapshot data available")


## 9) Journal + stop-loss diagnostics

Use fills and ledger to inspect de-risking behavior. In long-only runs, clusters of SELL fills after adverse moves can indicate stop-loss liquidations, while delayed re-buys reflect cooldown.


In [ ]:
if not journal_df.empty:
    summary = journal_df.groupby("entry_type", as_index=False)["amount"].sum().sort_values("amount")
    display(summary)

    if "BORROW_COST" in set(journal_df["entry_type"]):
        borrow = journal_df[journal_df["entry_type"] == "BORROW_COST"].copy()
        borrow = borrow.sort_values("timestamp")
        borrow["cum_borrow"] = borrow["amount"].cumsum()

        ax = borrow.plot(x="timestamp", y="cum_borrow", figsize=(12, 4), legend=False)
        ax.set_title("Cumulative Borrow Cost")
        ax.set_ylabel("Amount")
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
else:
    print("No journal entries available")

if not fills_df.empty:
    sell_fills = fills_df[fills_df["side"] == "SELL"].sort_values("timestamp")
    print(f"SELL fills: {len(sell_fills)} / total fills: {len(fills_df)}")
    if not sell_fills.empty:
        display(sell_fills.tail(15))
else:
    print("No fills available")


## 10) Case study takeaways template

In [ ]:

highlights = {
    "Num fills": len(fills_df),
    "Num journal entries": len(journal_df),
    "Final equity": float(result.equity_curve[-1]) if result.equity_curve else np.nan,
    "Max drawdown": float(metrics.get("MaxDrawdown", np.nan)),
    "Sharpe": float(metrics.get("Sharpe", np.nan)),
    "Total turnover": float(metrics.get("Turnover", np.nan)),
}

pd.Series(highlights)
